# IFN680 Assessment 3 - Aircraft Classification

**Student:** Karan Rooprai  **Student ID:** N12498122

**Student:** [Hieu]  **Student ID:** [Hieu]

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms



import tqdm
import copy
import random

import pickle


torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') #this line checks if we have a GPU available
print(f"Using device: {device}")

Using device: cuda:0


## 2. Data

In [2]:
# Load the full training dataset first (without transforms for now)
trainval_dataset = torchvision.datasets.ImageFolder('FGVCAircraft_Subset20/trainval')
test_dataset = torchvision.datasets.ImageFolder('FGVCAircraft_Subset20/test')

print(f'Trainval dataset size: {len(trainval_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')

print(f"Classes ({num_classes}): {class_names}")
print(f"Trainval images: {len(trainval_dataset)}")
print(f"Test images:     {len(test_dataset)}")

# Plot one random sample from each class in a horizontal stripe
num_classes = 20
fig, axes = plt.subplots(2, 10, figsize=(20, 6))

# Get class names
class_names = trainval_dataset.classes
dataset_labels = np.array([label for _, label in trainval_dataset.samples])

# For each class, find one sample and plot it
for class_idx in range(num_classes):
    class_indices = np.where(dataset_labels == class_idx)[0] 
    
    # Pick a random sample from this class
    sample_idx = np.random.choice(class_indices)
    
    # Load and display the image
    img, _ = trainval_dataset[sample_idx]

    row, col = int(class_idx // 10), int(class_idx % 10)
    axes[row, col].imshow(img)
    axes[row, col].set_title(class_names[class_idx].split('-')[-1], fontsize=10)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()


Trainval dataset size: 1331
Test dataset size: 669


NameError: name 'num_classes' is not defined

### 2.1 Stratified train / validation split

We hold out 20% of `trainval` as a validation set, stratified by class so every aircraft type
keeps the same proportion in both splits. The same `random_state` is reused everywhere, so all
experiments see identical train/val images and differences come only from the design change
under test.

In [ ]:

def plot_class_distribution(*datasets, dataset_names=None, figsize=(14, 6)):
    """
    Plot the distribution of samples per class for multiple datasets.
    
    Args:
        *datasets: Variable number of ImageFolder datasets or Subsets
        dataset_names: List of names for each dataset (optional)
        figsize: Figure size (width, height)
    """
    
    if dataset_names is None:
        dataset_names = [f'dataset_{i}' for i in range(len(datasets))]
    
    # Collect data from all datasets
    df_list = []
    for dataset, name in zip(datasets, dataset_names):
        # Handle both ImageFolder and Subset datasets
        if hasattr(dataset, 'samples'):
            # ImageFolder dataset
            labels_full = [label for _, label in dataset.samples]
            class_names = dataset.classes
        else:
            # Subset dataset - get labels from indices
            labels_full = [dataset.dataset.targets[i] for i in dataset.indices]
            class_names = dataset.dataset.classes
        
        df = pd.DataFrame({
            'class': [class_names[label] for label in labels_full],
            'split': name,
            'count': 1
        })
        df_list.append(df)
    
    # Combine all dataframes
    combined_df = pd.concat(df_list)
    
    # Group by class and split, then count
    df_grouped = combined_df.groupby(['class', 'split']).count().reset_index()
    
    # Create grouped bar plot
    plt.figure(figsize=figsize)
    sns.barplot(data=df_grouped, x='class', y='count', hue='split')
    plt.xticks(rotation=45, ha='right')
    plt.xlabel('Class')
    plt.ylabel('Number of Samples')
    plt.title('Distribution of Samples per Class')
    plt.tight_layout()
    plt.show()

# Plot the distribution
plot_class_distribution(trainval_dataset, test_dataset, dataset_names=['TrainVal', 'Test'])

In [ ]:
indices = list(range(len(trainval_dataset)))
labels  = [label for _, label in trainval_dataset.samples]

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,      # keep class balance in both splits
    random_state=42,
)
print(f"Train samples: {len(train_indices)}   Val samples: {len(val_indices)}")


train_dataset = torch.utils.data.Subset(trainval_dataset, train_indices)
val_dataset = torch.utils.data.Subset(copy.deepcopy(trainval_dataset), val_indices)

print(f"Total trainval samples: {len(trainval_dataset)}")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
plot_class_distribution(train_dataset, val_dataset, dataset_names=['Train', 'Val'])

### 2.2 Preprocessing and augmentation transforms


In [ ]:
imagenet_means = (0.485, 0.456, 0.406)
imagenet_stds  = (0.229, 0.224, 0.225)

# Plain preprocessing (no augmentation)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224)),
    transforms.Normalize(imagenet_means, imagenet_stds),
])

# Preprocessing WITH augmentation (used only for training in the relevant experiments)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    
    # transforms.ToTensor(),
    # transforms.Resize((224, 224)),
    # transforms.Normalize(imagenet_means, imagenet_stds),
])

# Apply transforms to the datasets
train_dataset.dataset.transform = transforms.Compose([train_transform,transform])
val_dataset.dataset.transform = transform
test_dataset.transform = transform


# create dataloaders for train, val, test datasets
batch_size = 32
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers = 0)
valloader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers = 0)
testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers = 0)

# Visualize a batch of augmented training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Plot the batch
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
axes = axes.ravel()
for idx in range(min(16, len(images))):
    # Denormalize the image for visualization
    img = images[idx].numpy().transpose((1, 2, 0))
    # just remap for visualization
    img = img * np.array(imagenet_stds) + np.array(imagenet_means)
    img = np.clip(img, 0, 1)
    
    axes[idx].imshow(img)
    axes[idx].set_title(class_names[labels[idx]])
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 3. Model and training helpers


In [ ]:
def setup_model(model, num_classes, freeze_backbone = False):
    
    # in_features = model.fc.in_features
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    if freeze_backbone: 
        for param in model.parameters():
            param.requires_grad = False
        
        for param in model.fc.parameters():
            param.requires_grad = True

    return model

backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
resnet_frozen = setup_model(backbone, 20, True)
print(resnet_frozen)

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, epoch, device):
    model.train() 
    train_loss, correct, total = [], 0.0, 0.0
    for _, data in  tqdm.tqdm(enumerate(dataloader, 0), total = len(dataloader), desc = f'Epoch {epoch+1} - training phase'):
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)        
        loss.backward()        
        optimizer.step()
        train_loss += [loss.cpu().item()]
        predicted = torch.argmax(outputs, axis = 1)        
        correct += torch.sum(predicted == labels).cpu().item()
        total += len(labels)
    mean_train_loss = np.mean(train_loss)
    train_accuracy = correct/total
    print(f"Training {epoch+1}: loss={mean_train_loss:.3f} acc={train_accuracy:.3f}")
    return mean_train_loss, train_accuracy


In [ ]:
def eval_epoch(model, dataloader, criterion, epoch, device):
    model.eval()
    with torch.no_grad(): 
        val_loss, val_correct, val_total = 0.0, 0.0, 0.0
        for i, data in  tqdm.tqdm(enumerate(dataloader, 0), total = len(dataloader), desc = f'Epoch {epoch+1} - validation phase'):
            inputs, labels = data            
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            predicted = torch.argmax(outputs, axis = 1)
            loss = criterion(outputs, labels)
            val_loss += loss.cpu().item() * inputs.size(0)            
            val_correct += torch.sum(predicted == labels).cpu().item()
            val_total += inputs.size(0)
    
    mean_val_loss = val_loss / val_total
    val_accuracy = val_correct / val_total
    print(f"Validation {epoch+1}: loss={mean_val_loss:.3f} acc={val_accuracy:.3f}")
    return mean_val_loss, val_accuracy

In [ ]:
def run_training(model, train_loader, val_loader, optimizer, criterion,
                 epochs, scheduler=None, save_path=None):
    """Train for `epochs`, recording learning curves. Returns (history, best_val_acc).

    If a val_loader is given, the best model (by val accuracy) is kept and optionally saved.
    If val_loader is None (final retrain on full trainval), the last-epoch model is saved.
    """
    model = model.to(device)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc, best_state = -1.0, None

    for epoch in range(epochs):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)

        if val_loader is not None:
            va_loss, va_acc = eval_epoch(model, val_loader, criterion, device)
            history['val_loss'].append(va_loss)
            history['val_acc'].append(va_acc)
            if va_acc > best_val_acc:
                best_val_acc = va_acc
                best_state = copy.deepcopy(model.state_dict())
            print(f"Epoch {epoch+1:2d}/{epochs} | "
                  f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
                  f"val loss {va_loss:.3f} acc {va_acc:.3f}")
        else:
            print(f"Epoch {epoch+1:2d}/{epochs} | train loss {tr_loss:.3f} acc {tr_acc:.3f}")

        if scheduler is not None:
            scheduler.step()

    # For the final retrain (no val set) keep the last-epoch weights
    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())
    if save_path is not None:
        torch.save(best_state, save_path)
        print(f"Saved weights -> {save_path}")

    history['best_val_acc'] = best_val_acc
    return history, best_val_acc

In [ ]:
# Collect every learning-curve history here so main_report.ipynb can reload and plot them
histories = {}
criterion = nn.CrossEntropyLoss()
EPOCHS = 12   # epochs per baseline/experiment run

## 4. Baseline — frozen ResNet-18 feature extractor

The reference point. ResNet-18 is loaded with ImageNet weights, the backbone is **frozen**, and
only the new 20-way head is trained with plain SGD (lr = 0.001, momentum = 0.9) on
**un-augmented** images. Its best validation accuracy is the *floor* every experiment must beat.

In [ ]:
backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
baseline_model = setup_model(backbone, num_classes, freeze_backbone=True)

train_loader, val_loader = make_loaders(eval_transform)   # no augmentation
optimizer = optim.SGD(filter(lambda p: p.requires_grad, baseline_model.parameters()),
                      lr=0.001, momentum=0.9)

histories['baseline'], _ = run_training(
    baseline_model, train_loader, val_loader, optimizer, criterion, epochs=EPOCHS)

## 5. Experiment 1 — Data augmentation

**Hypothesis:** randomly flipping, rotating, and colour-jittering the training images exposes the
frozen feature extractor to more varied views of each aircraft, reducing over-fitting on the small
training set and improving validation accuracy.

**Controlled change vs baseline:** the *only* difference is the training transform
(`aug_transform` instead of `eval_transform`). Architecture, freezing, optimiser and epochs are
identical.

In [ ]:
backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
aug_model = setup_model(backbone, num_classes, freeze_backbone=True)

train_loader, val_loader = make_loaders(aug_transform)    # augmentation ON
optimizer = optim.SGD(filter(lambda p: p.requires_grad, aug_model.parameters()),
                      lr=0.001, momentum=0.9)

histories['aug'], _ = run_training(
    aug_model, train_loader, val_loader, optimizer, criterion, epochs=EPOCHS)

## 6. Experiment 2 — Fine-tuning the backbone

**Hypothesis:** ImageNet features are generic; letting the whole network (not just the head) adapt
to aircraft should capture the fine-grained differences between variants and lift validation
accuracy above the frozen baseline.

**Controlled change vs baseline:** the *only* difference is that the backbone is **unfrozen**
(`freeze_backbone=False`), so every layer is trainable. The optimiser, learning rate (SGD,
lr = 0.001, momentum = 0.9), epochs, and (un-augmented) data are kept **identical** to the
baseline, so any change in accuracy is attributable to fine-tuning alone.

In [ ]:
backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
finetune_model = setup_model(backbone, num_classes, freeze_backbone=False)  # all layers trainable

train_loader, val_loader = make_loaders(eval_transform)   # no augmentation
# Same optimiser + learning rate as the baseline -> the ONLY changed variable is freeze->unfreeze
optimizer = optim.SGD(finetune_model.parameters(), lr=0.001, momentum=0.9)

histories['finetune'], _ = run_training(
    finetune_model, train_loader, val_loader, optimizer, criterion, epochs=EPOCHS)

## 7. Experiment 3 — AdamW optimiser + Cosine-Annealing schedule

**Hypothesis:** replacing plain SGD with **AdamW** (adaptive updates + weight decay) and decaying
the learning rate along a **cosine** curve should give faster, more stable convergence of the
classification head and a small accuracy gain over the SGD baseline.

**Controlled change vs baseline:** only the optimiser and schedule change. The backbone stays
frozen and images stay un-augmented, so any difference is attributable to the optimisation
strategy.

In [ ]:
backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
adamw_model = setup_model(backbone, num_classes, freeze_backbone=True)

train_loader, val_loader = make_loaders(eval_transform)   # no augmentation
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, adamw_model.parameters()),
                        lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

histories['adamw'], _ = run_training(
    adamw_model, train_loader, val_loader, optimizer, criterion,
    epochs=EPOCHS, scheduler=scheduler)

## 8. Experiment comparison

Best validation accuracy for the baseline and each experiment. The experiments that beat the
baseline are the ingredients we combine into the final model.

In [ ]:
print(f"{'Model':<28}{'Best val accuracy':>18}")
print('-' * 46)
for name, label in [('baseline', 'Baseline (frozen SGD)'),
                    ('aug',      'Exp 1: + Augmentation'),
                    ('finetune', 'Exp 2: Fine-tuning'),
                    ('adamw',    'Exp 3: AdamW + Cosine')]:
    print(f"{label:<28}{histories[name]['best_val_acc']:>18.4f}")

## 9. Best model — combine the winners, retrain on all `trainval`

The final model combines only the changes that **improved** validation accuracy over the baseline.
Fine-tuning (+0.25) and AdamW + Cosine-Annealing (+0.04) both helped, so both are kept.
Augmentation is **excluded**: it *reduced* validation accuracy for the frozen model (−0.10), so it
is not a winning ingredient and is dropped rather than carried into the final model.

The best model is therefore an **unfrozen** ResNet-18 trained with **AdamW + Cosine-Annealing** on
**un-augmented** images. Unlike the controlled experiments, this is a deliberately combined
configuration, so the learning rate is set to a smaller value (0.0001) suited to fine-tuning the
whole network with AdamW. As required, it is retrained on the **entire** `trainval` set (no
validation split) so it uses every labelled image before facing the test set. A slightly longer
schedule (15 epochs) is used, and the weights are saved to `best_model.pth`.

In [ ]:
FINAL_EPOCHS = 15

backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
best_model = setup_model(backbone, num_classes, freeze_backbone=False)  # fine-tune (a winner)

# Winners only: fine-tuning + AdamW/cosine. Augmentation is dropped (it hurt the baseline),
# so the final model trains on un-augmented images (eval_transform).
full_train_loader, _ = make_loaders(eval_transform, full_trainval=True)  # all trainval, no augmentation
optimizer = optim.AdamW(best_model.parameters(), lr=0.0001, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINAL_EPOCHS)

histories['best'], _ = run_training(
    best_model, full_train_loader, None, optimizer, criterion,
    epochs=FINAL_EPOCHS, scheduler=scheduler, save_path='best_model.pth')

## 10. Save all learning curves

`histories.pkl` stores every learning curve and validation score. `main_report.ipynb` reloads it
(together with `best_model.pth`) to reproduce all figures, the comparison table, and the final
test metric without retraining.

In [ ]:
with open('histories.pkl', 'wb') as f:
    pickle.dump({'histories': histories, 'class_names': class_names}, f)
print("Saved histories.pkl")
print("Saved files:", [x for x in os.listdir('.') if x.endswith(('.pkl', '.pth'))])